In [1]:
import polars as pl
import os

In [2]:
DATASET_PATH = '/group/pmc021/amunif/epi-thesis/workflow/15_HepG2 from ENCODE/dataset/'

In [3]:
# Function to read histone file and returns the data frame
def load_histone(filename):
    # Declare the intersect file schema
    # Define the schema for ensembl and gappedPeak dataset
    intersect_schema = pl.Schema({
        # The ensembl_top1.csv columns
        'chrom': pl.String,
        'tss_minus_5k': pl.Int64,
        'tss_plus_5k': pl.Int64, 
        'test_id': pl.String,
        'gene_id': pl.String,
        'chrom_name': pl.String,
        'gene_start': pl.Int64,
        'gene_end': pl.Int64,
        'strand': pl.Int64,
        'gene_name': pl.String,
        'gene_stable_id': pl.String,
        'gene_type': pl.String,
        'tss': pl.Int64,
        'locus': pl.String,
        'orig_start': pl.Int64, 
        'orig_end': pl.Int64,
        
        # gappedPeak columns
        "chrom_p": pl.String,
        "chromStart_p": pl.Int64, 
        "chromEnd_p": pl.Int64, 
        "name_p": pl.String, 
        "score_p": pl.Float64, 
        "strand_p": pl.String,
        "thickStart": pl.Int64,
        "thickEnd": pl.Int64,
        "itemRgb": pl.Int64,
        "blockCount": pl.Int64,
        "blockSizes": pl.String,
        "blockStarts": pl.String,
        "signalValue": pl.Float64,
        "pValue": pl.Float64,
        "qValue": pl.Float64
    })

    # Open file
    histone_df = pl.read_csv(
            filename,
            separator="\t",
            has_header = False,
            schema = intersect_schema   
        )
    
    return histone_df

In [4]:
def create_empty_dataframe(histone_name):
    schema = pl.Schema({
        'gene_id': pl.String,
        histone_name: pl.List(pl.Float64),
        f'{histone_name}_wc': pl.UInt32,
        f'{histone_name}_len': pl.UInt32
    })

    df = pl.DataFrame(schema=schema)

    return df

In [5]:
def build_matrix(genes_df, histone_df, histone_name):
    # Build the dataframe with window
    genes_with_windows = genes_df.with_columns([
        pl.int_ranges(pl.col('tss_minus_5k'), pl.col('tss_plus_5k'), 100).alias('window_start')
    ]).explode('window_start')

    genes_with_windows = genes_with_windows.with_columns([
            (pl.col('window_start') + 100).alias('window_end')
    ])

    # Join genes with histone data
    joined_df = genes_with_windows.join(
        histone_df,
        left_on='gene_id',
        right_on='gene_id',
        how='left'
    )

    # Filter and calculate average signal value
    result_df = joined_df.filter(
        (pl.col('chromStart_p') < pl.col('window_end')) &
        (pl.col('chromEnd_p') > pl.col('window_start'))
    ).group_by(['gene_id', 'window_start'], maintain_order=True).agg([
        pl.col('signalValue').mean().alias(histone_name)
    ]).sort(['gene_id', 'window_start'])

    # Find the gene without histone match
    genes_wo_histone = genes_with_windows.join(
        result_df,
        on=["gene_id", "window_start"],
        how="anti"
    )

    # Add the signalValue column so it can be merged
    genes_wo_histone = genes_wo_histone.with_columns(
        signalValue = pl.lit(0.0).cast(pl.Float64)
    )

    # Aggregate the genes without histone result
    genes_wo_histone = genes_wo_histone.group_by(['gene_id', 'window_start'], maintain_order=True).agg([
            pl.col('signalValue').mean().alias(histone_name)
        ]).sort(['gene_id', 'window_start'])

    # Merge both (results and genes without histone)
    result_df.extend(genes_wo_histone)

    # Sort the result dataframe by gene_id and window start for aggregation
    sorted_result_df = result_df.sort(['gene_id', 'window_start'])
    
    # Group by to make array of features
    matrix_df = (
        sorted_result_df
        .with_columns(pl.col(histone_name).fill_null(0))
        .group_by(['gene_id'], maintain_order=True)
        .agg(pl.col(histone_name))
        .sort('gene_id')
    )

    # Add the count and length of array feature for checking
    matrix_control_df = matrix_df.with_columns(
        pl.col(histone_name)
        .list.eval(pl.element().is_not_null() & (pl.element() > 0))
        .list.sum()
        .alias(f"{histone_name}_wc")
    )

    matrix_control_df = matrix_control_df.with_columns(
        pl.col(histone_name).list.len().alias(f"{histone_name}_len")
    )

    # Finally, return the gene_id with its histone features
    return matrix_control_df

In [6]:
# Getting histone in chunk
def get_histone_features(genes_df, histone_df, histone_name):
    
    genes_w_histone = create_empty_dataframe(histone_name)
    
    for i, chunk in enumerate(genes_df.iter_slices(n_rows=100)):
        if i % 100 == 0:
            print(f"Processing {histone_name}: {i*100}/{genes_df.height}")
        
        result = build_matrix(chunk, histone_df, histone_name)
        genes_w_histone.extend(result)

    print(f"Processing {histone_name} features finished.")
    return genes_w_histone

In [7]:
# Load the HepG2 expression file
genes_df = pl.read_parquet(os.path.join(DATASET_PATH, 'ensembl_HepG2exp.parquet'))

In [8]:
genes_df

chrom,tss_minus_5k,tss_plus_5k,test_id,gene_id,chrom_name,gene_start,gene_end,strand,gene_name,gene_stable_id,gene_type,tss,locus,orig_start,orig_end,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant
str,i64,i64,str,str,str,i64,i64,i64,str,str,str,i64,str,i64,i64,str,f64,f64,f64,f64,f64,f64,str
"""chr1""",64091,74091,"""XLOC_000001""","""XLOC_000001""","""1""",69091,70008,1,"""OR4F5""","""ENSG00000186092""","""protein_coding""",69091,"""chr1:69090-70008""",69090,70008,"""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no"""
"""chr1""",362640,372640,"""XLOC_000003""","""XLOC_000003""","""1""",367640,368634,1,"""OR4F29""","""ENSG00000235249""","""protein_coding""",367640,"""chr1:367658-368597""",367658,368597,"""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no"""
"""chr1""",855260,865260,"""XLOC_000006""","""XLOC_000006""","""1""",860260,879955,1,"""SAMD11""","""ENSG00000187634""","""protein_coding""",860260,"""chr1:851135-917473""",851135,917473,"""NOTEST""",0.0888452,0.136965,0.624439,0.0,1.0,1.0,"""no"""
"""chr1""",855260,865260,"""XLOC_000007""","""XLOC_000007""","""1""",860260,879955,1,"""SAMD11""","""ENSG00000187634""","""protein_coding""",860260,"""chr1:851135-917473""",851135,917473,"""OK""",4.04743,4.16567,0.041543,0.0707864,0.94475,0.999565,"""no"""
"""chr1""",943803,953803,"""XLOC_000008""","""XLOC_000008""","""1""",948803,949920,1,"""ISG15""","""ENSG00000187608""","""protein_coding""",948803,"""chr1:948846-949919""",948846,949919,"""OK""",26.7934,30.2747,0.176232,0.797534,0.42445,0.999565,"""no"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",24324129,24334129,"""XLOC_030009""","""XLOC_030009""","""Y""",24314689,24329129,-1,"""RBMY1F""","""ENSG00000169800""","""protein_coding""",24329129,"""chrY:24314688-24329089""",24314688,24329089,"""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no"""
"""chrY""",25340241,25350241,"""XLOC_030012""","""XLOC_030012""","""Y""",25275502,25345241,-1,"""DAZ1""","""ENSG00000188120""","""protein_coding""",25345241,"""chrY:25275501-25345239""",25275501,25345239,"""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no"""
"""chrY""",26189166,26199166,"""XLOC_030014""","""XLOC_030014""","""Y""",26191376,26194166,-1,"""CDY1B""","""ENSG00000172352""","""protein_coding""",26194166,"""chrY:26191376-26194161""",26191376,26194161,"""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no"""


In [9]:
# Load the histone dataset
H3K4me3_df = load_histone(os.path.join(DATASET_PATH, 'H3K4me3.bed'))
H3K9ac_df = load_histone(os.path.join(DATASET_PATH, 'H3K9ac.bed'))
H3K9me3_df = load_histone(os.path.join(DATASET_PATH, 'H3K9me3.bed'))
H3K27ac_df = load_histone(os.path.join(DATASET_PATH, 'H3K27ac.bed'))
H3K27me3_df = load_histone(os.path.join(DATASET_PATH, 'H3K27me3.bed'))

In [11]:
# Build the matrix
genes_w_H3K4me3_df = get_histone_features(genes_df, H3K4me3_df, 'H3K4me3')
genes_w_H3K9ac_df = get_histone_features(genes_df, H3K9ac_df, 'H3K9ac')
genes_w_H3K9me3_df = get_histone_features(genes_df, H3K9me3_df, 'H3K9me3')
genes_w_H3K27ac_df = get_histone_features(genes_df, H3K27ac_df, 'H3K27ac')
genes_w_H3K27me3_df = get_histone_features(genes_df, H3K27me3_df, 'H3K27me3')

Processing H3K4me3: 0/22154
Processing H3K4me3: 10000/22154
Processing H3K4me3: 20000/22154
Processing H3K4me3 features finished.
Processing H3K9ac: 0/22154
Processing H3K9ac: 10000/22154
Processing H3K9ac: 20000/22154
Processing H3K9ac features finished.
Processing H3K9me3: 0/22154
Processing H3K9me3: 10000/22154
Processing H3K9me3: 20000/22154
Processing H3K9me3 features finished.
Processing H3K27ac: 0/22154
Processing H3K27ac: 10000/22154
Processing H3K27ac: 20000/22154
Processing H3K27ac features finished.
Processing H3K27me3: 0/22154
Processing H3K27me3: 10000/22154
Processing H3K27me3: 20000/22154
Processing H3K27me3 features finished.


# Checking the generated features

In [12]:
genes_w_H3K9ac_df.filter(pl.col('H3K9ac_wc') > 0).sort(['H3K9ac_wc'], descending=True)

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len
str,list[f64],u32,u32
"""XLOC_016670""","[14.1102, 14.1102, … 14.1102]",100,100
"""XLOC_017014""","[14.1102, 14.1102, … 14.1102]",100,100
"""XLOC_018923""","[9.61892, 9.61892, … 9.61892]",100,100
"""XLOC_022007""","[14.7424, 14.7424, … 14.7424]",100,100
"""XLOC_026220""","[20.2331, 20.2331, … 20.2331]",100,100
…,…,…,…
"""XLOC_018455""","[0.0, 0.0, … 3.33662]",1,100
"""XLOC_022433""","[3.77713, 0.0, … 0.0]",1,100
"""XLOC_023287""","[0.0, 0.0, … 3.42963]",1,100


In [13]:
genes_w_H3K9ac_df.filter(pl.col('H3K9ac_len') < 100).sort(['H3K9ac_wc'], descending=True)

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len
str,list[f64],u32,u32


In [14]:
genes_w_H3K9me3_df.filter(pl.col('H3K9me3_wc') > 0).sort(['H3K9me3_wc'], descending=True)

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32
"""XLOC_013483""","[4.262, 4.262, … 4.262]",100,100
"""XLOC_013600""","[3.86704, 3.86704, … 3.86704]",100,100
"""XLOC_017426""","[4.52623, 4.52623, … 5.29128]",100,100
"""XLOC_022697""","[5.78071, 5.78071, … 5.78071]",100,100
"""XLOC_025821""","[5.19098, 5.19098, … 5.19098]",100,100
…,…,…,…
"""XLOC_025749""","[0.0, 0.0, … 3.24647]",1,100
"""XLOC_026502""","[0.0, 0.0, … 2.22752]",1,100
"""XLOC_027685""","[2.94969, 0.0, … 0.0]",1,100


In [15]:
genes_w_H3K9me3_df.filter(pl.col('H3K9me3_len') < 100).sort(['H3K9me3_wc'], descending=True)

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32


In [16]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_wc') > 0).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32
"""XLOC_026220""","[25.7556, 25.7556, … 25.7556]",100,100
"""XLOC_012851""","[0.0, 0.0, … 16.1295]",98,100
"""XLOC_000349""","[6.04266, 6.04266, … 0.0]",94,100
"""XLOC_001588""","[6.04266, 6.04266, … 0.0]",94,100
"""XLOC_012852""","[16.1295, 16.1295, … 17.9676]",94,100
…,…,…,…
"""XLOC_029276""","[9.21696, 0.0, … 0.0]",1,100
"""XLOC_029277""","[9.21696, 0.0, … 0.0]",1,100
"""XLOC_029278""","[9.21696, 0.0, … 0.0]",1,100


In [17]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_len') < 100).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32


In [18]:
genes_w_H3K27ac_df.filter(pl.col('H3K27ac_wc') > 0).sort(['H3K27ac_wc'], descending=True)

gene_id,H3K27ac,H3K27ac_wc,H3K27ac_len
str,list[f64],u32,u32
"""XLOC_026926""","[23.9515, 23.9515, … 23.9515]",100,100
"""XLOC_016670""","[24.5305, 24.5305, … 0.0]",99,100
"""XLOC_017014""","[24.5305, 24.5305, … 0.0]",99,100
"""XLOC_022699""","[23.6256, 23.6256, … 18.3577]",99,100
"""XLOC_026220""","[11.2074, 11.2074, … 19.781]",99,100
…,…,…,…
"""XLOC_014022""","[0.0, 0.0, … 5.00209]",1,100
"""XLOC_022418""","[3.23558, 0.0, … 0.0]",1,100
"""XLOC_023249""","[17.5295, 0.0, … 0.0]",1,100


In [19]:
genes_w_H3K27ac_df.filter(pl.col('H3K27ac_len') < 100).sort(['H3K27ac_wc'], descending=True)

gene_id,H3K27ac,H3K27ac_wc,H3K27ac_len
str,list[f64],u32,u32


In [20]:
genes_w_H3K27me3_df.filter(pl.col('H3K27me3_wc') > 0).sort(['H3K27me3_wc'], descending=True)

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32
"""XLOC_008117""","[4.04405, 4.04405, … 4.04405]",100,100
"""XLOC_008118""","[4.04405, 4.04405, … 4.04405]",100,100
"""XLOC_011036""","[4.74379, 4.74379, … 3.45311]",98,100
"""XLOC_011681""","[4.74379, 4.74379, … 3.45311]",98,100
"""XLOC_012222""","[4.74379, 4.74379, … 3.45311]",98,100
…,…,…,…
"""XLOC_023638""","[3.16259, 0.0, … 0.0]",1,100
"""XLOC_025393""","[2.80467, 0.0, … 0.0]",1,100
"""XLOC_025922""","[0.0, 0.0, … 2.52149]",1,100


In [21]:
genes_w_H3K27me3_df.filter(pl.col('H3K27me3_len') < 100).sort(['H3K27me3_wc'], descending=True)

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32


# Join all histones into single data frame

In [22]:
# Join all histone into single dataframe
genes_histone_df = genes_w_H3K9ac_df \
                    .join(genes_w_H3K9me3_df, on='gene_id') \
                    .join(genes_w_H3K4me3_df, on='gene_id') \
                    .join(genes_w_H3K27ac_df, on='gene_id') \
                    .join(genes_w_H3K27me3_df, on='gene_id')

In [23]:
genes_histone_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",42,100,"[0.0, 0.0, … 0.0]",0,100,"[4.9485, 4.9485, … 4.04299]",83,100
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",42,100,"[0.0, 0.0, … 0.0]",0,100,"[4.9485, 4.9485, … 4.04299]",83,100
"""XLOC_000008""","[0.0, 0.0, … 0.0]",24,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",26,100,"[0.0, 0.0, … 0.0]",17,100,"[0.0, 0.0, … 0.0]",5,100
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100


# Join genes with the value and histone features

In [24]:
genes_values_df = genes_df.select(['gene_id', 'value_1', 'value_2'])
genes_values_df

gene_id,value_1,value_2
str,f64,f64
"""XLOC_000001""",0.0,0.0
"""XLOC_000003""",0.0,0.0
"""XLOC_000006""",0.0888452,0.136965
"""XLOC_000007""",4.04743,4.16567
"""XLOC_000008""",26.7934,30.2747
…,…,…
"""XLOC_030009""",0.0,0.0
"""XLOC_030012""",0.0,0.0
"""XLOC_030014""",0.0,0.0


In [25]:
genes_histone_values_df = genes_histone_df.join(
    genes_values_df,
    on = 'gene_id'
)

In [26]:
genes_histone_values_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,value_1,value_2
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64,f64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",42,100,"[0.0, 0.0, … 0.0]",0,100,"[4.9485, 4.9485, … 4.04299]",83,100,0.0888452,0.136965
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",42,100,"[0.0, 0.0, … 0.0]",0,100,"[4.9485, 4.9485, … 4.04299]",83,100,4.04743,4.16567
"""XLOC_000008""","[0.0, 0.0, … 0.0]",24,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",26,100,"[0.0, 0.0, … 0.0]",17,100,"[0.0, 0.0, … 0.0]",5,100,26.7934,30.2747
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0


In [27]:
# Save to parquet
genes_histone_values_df.write_parquet(os.path.join(DATASET_PATH, 'HepG2_exp_histones.parquet'))

In [28]:
test_df = pl.read_parquet(os.path.join(DATASET_PATH, 'HepG2_exp_histones.parquet'))
test_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,value_1,value_2
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64,f64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",42,100,"[0.0, 0.0, … 0.0]",0,100,"[4.9485, 4.9485, … 4.04299]",83,100,0.0888452,0.136965
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",42,100,"[0.0, 0.0, … 0.0]",0,100,"[4.9485, 4.9485, … 4.04299]",83,100,4.04743,4.16567
"""XLOC_000008""","[0.0, 0.0, … 0.0]",24,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",26,100,"[0.0, 0.0, … 0.0]",17,100,"[0.0, 0.0, … 0.0]",5,100,26.7934,30.2747
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
